In [1]:
import telemetry_parser
import pandas as pd
import numpy as np
import math

In [ ]:
path = ''
video = 'vidD.mp4'

**IMU CSV**

In [3]:
tp = telemetry_parser.Parser(path + video)
print('Camera: ', tp.camera)
print('Model: ', tp.model)

# return all telemetry as an array of dicts
telemetry = tp.telemetry()

# format the values with units etc
telemetry_format = tp.telemetry(human_readable = True)

# return only gyro and accel with timestamps, normalized to a single orientation and scaled to deg/s and m/s2
norm_IMU = tp.normalized_imu()

Camera:  Insta360
Model:  Insta360 X3


In [4]:
# Convert telemetry to DataFrame
df = pd.DataFrame(norm_IMU)

# Expand tuples
df[['gyro_x_deg', 'gyro_y_deg', 'gyro_z_deg']] = pd.DataFrame(df['gyro'].tolist(), index=df.index)
df[['accel_x', 'accel_y', 'accel_z']] = pd.DataFrame(df['accl'].tolist(), index=df.index)

# Drop original tuple columns
df.drop(columns=['gyro', 'accl', 'magn'], inplace=True)

# Keep only non-negative timestamps
df = df[df['timestamp_ms'] >= 0]

# Convert timestamp ms → ns
#df['timestamp_ns'] = (df['timestamp_ms'] * 1e6).astype('int64')

# Gyro deg/s → rad/s
deg2rad = math.pi / 180.0
df['w_RS_S_x'] = df['gyro_x_deg'] * deg2rad
df['w_RS_S_y'] = df['gyro_y_deg'] * deg2rad
df['w_RS_S_z'] = df['gyro_z_deg'] * deg2rad

# Accel stays in m/s²
df['a_RS_S_x'] = df['accel_x']
df['a_RS_S_y'] = df['accel_y']
df['a_RS_S_z'] = df['accel_z']

# Arrange columns exactly as expected
df_out = df[['timestamp_ms',
             'w_RS_S_x', 'w_RS_S_y', 'w_RS_S_z',
             'a_RS_S_x', 'a_RS_S_y', 'a_RS_S_z']]

# Proper header with units (single line, no extra column names)
header_line = "#timestamp [ms],w_RS_S_x [rad s^-1],w_RS_S_y [rad s^-1],w_RS_S_z [rad s^-1],a_RS_S_x [m s^-2],a_RS_S_y [m s^-2],a_RS_S_z [m s^-2]"

# Write output in EXACT same style
with open("IMUdata.csv", "w") as f:
    f.write(header_line + "\n")
df_out.to_csv("IMUdata.csv", index=False, mode='a', header=False)

print("Saved IMU data → IMUdata.csv")
print(df_out.head())

Saved IMU data → IMUdata.csv
      timestamp_ms  w_RS_S_x  w_RS_S_y  w_RS_S_z  a_RS_S_x  a_RS_S_y  a_RS_S_z
1860         0.475 -0.048202 -0.102511 -0.071207  1.886822  9.607238  0.478063
1861         1.475 -0.047078 -0.105696 -0.068021  1.915371  9.607203  0.467957
1862         2.475 -0.044928 -0.107823 -0.066988  1.915194  9.607170  0.458382
1863         3.475 -0.044889 -0.107815 -0.064858  1.915194  9.626323  0.458316
1864         4.475 -0.045895 -0.106739 -0.061646  1.924946  9.626356  0.467714


**Images CSV**

In [10]:
tp = telemetry_parser.Parser(path + video)
print('Camera: ', tp.camera)
print('Model: ', tp.model)

# return all telemetry as an array of dicts
telemetry = tp.telemetry()

# format the values with units etc
telemetry_format = tp.telemetry(human_readable = True)

# return only gyro and accel with timestamps, normalized to a single orientation and scaled to deg/s and m/s2
norm_IMU = tp.normalized_imu()

Camera:  Insta360
Model:  Insta360 X3


In [5]:
# Assuming telemetry is already loaded
exposure_data = telemetry[0]['Exposure']['Data']

# Convert to DataFrame
df = pd.DataFrame(exposure_data)

# Filter rows where t >= 0
df = df[df['t'] >= 0].reset_index(drop=True)

# Convert t from seconds to milliseconds
df['t'] = df['t'] * 1e3  # now in milliseconds

# Create filename column
df['filename'] = df.index + 1
df['filename'] = df['filename'].astype(str) + ".png"

# Proper header with units (single line, no extra column names)
header_line = "#timestamp [ms],filename"

# Write output in EXACT same style
with open("Imagedata.csv", "w") as f:
    f.write(header_line + "\n")
    for _, row in df.iterrows():
        f.write(f"{row['t']:.6f},{row['filename']}\n")

print("Saved Image data → Imagedata.csv")

Saved Image data → Imagedata.csv


**Frames**

In [20]:
import cv2
import os


output_dir = 'images'           # directory to save frames

# --- Create output directory if it doesn't exist ---
os.makedirs('images', exist_ok=True)

# --- Load video ---
cap = cv2.VideoCapture(path + video)

if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

frame_idx = 1

while True:
    ret, frame = cap.read()
    if not ret:
        break  # end of video

    filename = f"{frame_idx}.png"
    filepath = os.path.join(output_dir, filename)
    cv2.imwrite(filepath, frame)
    frame_idx += 1

cap.release()
print(f"Saved {frame_idx - 1} frames to '{output_dir}/'")

Saved 2699 frames to 'images/'
